In [1]:
import wfdb     # reads MIT-BIH .dat/.hea/.atr files
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt       # for the bandpass filter (noise removal)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib       # for saving your trained model later
import os

In [2]:
data = 'mit-bih-arrhythmia-database'
# .hea files exist for every record, so use those to get unique IDs
all_files = os.listdir(data)
record_ids = sorted(set(f.split('.')[0] for f in all_files if f.endswith('.hea')))

print(f"Found {len(record_ids)} records:")
print(record_ids)

Found 48 records:
['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214', '215', '217', '219', '220', '221', '222', '223', '228', '230', '231', '232', '233', '234']


In [3]:
from features import bandpass_filter, extract_features_from_record

In [4]:
data_folder = data  # data = 'mit-bih-arrhythmia-database', from your earlier cell

all_dfs = []

for rec_id in record_ids:
    try:
        df,_,_ = extract_features_from_record(rec_id, data_folder=data_folder)
        all_dfs.append(df)
    except Exception as e:
        print(f"Skipped {rec_id}: {e}")

full_dataset = pd.concat(all_dfs, ignore_index=True)
print(f"Total beats across all records: {full_dataset.shape}")
print(full_dataset['true_label'].value_counts())

Total beats across all records: (112566, 7)
true_label
Normal      75052
Abnormal    37514
Name: count, dtype: int64


In [5]:
X = full_dataset[['rr_interval', 'heart_rate', 'hrv', 'amplitude', 'qrs_width']]
y = full_dataset['true_label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9488318379674869

Confusion Matrix:
[[ 6974   529]
 [  623 14388]]

Classification Report:
              precision    recall  f1-score   support

    Abnormal       0.92      0.93      0.92      7503
      Normal       0.96      0.96      0.96     15011

    accuracy                           0.95     22514
   macro avg       0.94      0.94      0.94     22514
weighted avg       0.95      0.95      0.95     22514



In [6]:
import joblib
joblib.dump(model, 'ecg_model.pkl')
print("Model saved.")

Model saved.
